In [77]:
import scipy as sp
import numpy as np
import numpy.linalg as la
import sympy as smp
import scipy.stats as stats
import matplotlib.pyplot as plt

import tensorly as tl

In [78]:
# tensor -> matrix

def unfold(M, mode=0):
    m = np.moveaxis(M, mode, 0).reshape(M.shape[mode], -1)
    return m

mat = np.array([
    [
        [1, 5],
        [3, 7]
    ],
    [
        [2, 6],
        [4, 8]
    ]
])

mat, unfold(mat, 2)


(array([[[1, 5],
         [3, 7]],
 
        [[2, 6],
         [4, 8]]]),
 array([[1, 3, 2, 4],
        [5, 7, 6, 8]]))

In [79]:
def factor_matrices(M):
    factors = []

    for mode in range(M.ndim):
        A = unfold(M, mode)
        U, _, _ = la.svd(A)
        factors.append(U)
    
    return factors

m_fac = factor_matrices(mat)
print(f"{[m.shape for m in m_fac]}")
m_fac

[(2, 2), (2, 2), (2, 2)]


[array([[-0.64142303, -0.7671874 ],
        [-0.7671874 ,  0.64142303]]),
 array([[-0.56672424, -0.82390754],
        [-0.82390754,  0.56672424]]),
 array([[-0.37616823, -0.92655138],
        [-0.92655138,  0.37616823]])]

In [80]:

def refold_0(tensor):
    return np.array([
        [[tensor[i, j + 2 * k] for j in range(2)] for k in range(2)] for i in range(2)
    ])

def refold(tensor, mode, shape):
    full_shape = [shape[mode]] + [shape[i] for i in range(len(shape)) if i != mode]
    return tensor.reshape(full_shape).swapaxes(0, mode)

def mode_n_product(T, U, mode):
    A = unfold(T, mode)
    result = U @ A
    
    # refold mat
    new_shape = list(T.shape)
    new_shape[mode] = U.shape[1]
    return result.reshape(
        [new_shape[mode]] + [new_shape[i] for i in range(len(new_shape)) if i != mode]
    ).swapaxes(0, mode)

def core_tensor(M, factors):
    G = M.copy()
    for mode, U in enumerate(factors):
        G = mode_n_product(G, U, mode)
    return G

G = core_tensor(mat, m_fac)
G

array([[[-1.42253953e+01,  4.61793060e-03],
        [ 1.60125603e-02,  5.43770692e-01]],

       [[ 8.28025332e-03,  1.11585148e+00],
        [ 2.38589095e-01,  2.00114739e-01]]])

In [81]:
def hosvd(M):
    factors = factor_matrices(M)
    G = core_tensor(M, factors)
    return G, factors


In [ ]:
tensor = np.array([
    [
        [0, 1],
        [1, 0]
    ],
    [
        [1, 0],
        [0, 1]
    ]
])

ct = hosvd(tensor)[0]
A = stats.ortho_group.rvs(2)

A_1234 = np.array([
    [
        [1, 2],
        [3, 4]
    ],
    [
        [5, 6],
        [7, 8]
    ]
])

A = np.array([
    [1, 2],
    [3, 4]
])

temp = mode_n_product(ct, A, 0)
hosvd(temp), ct
temp, A, ct



ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2,) + inhomogeneous part.

In [ ]:
t_f = unfold(A_1234)
t_rf = refold(t_f, mode=0, shape=(2, 2, 2))

t_f, t_rf

(array([[1, 2, 3, 4],
        [5, 6, 7, 8]]),
 array([[[1, 2],
         [3, 4]],
 
        [[5, 6],
         [7, 8]]]))

In [ ]:
unfold = tl.unfold(A_1234, mode=0)
fold = tl.fold(unfold, mode=0, shape=(2, 2, 2))
unfold, fold

(array([[1, 2, 3, 4],
        [5, 6, 7, 8]]),
 array([[[1, 2],
         [3, 4]],
 
        [[5, 6],
         [7, 8]]]))